# Pixel explorer: the study area (Sentinel-1 + Sentinel-2, 2025-09-01 to 2026-10-08)

Give a **pixel id** (band 1 of `data/grid/index.tif`) or lon/lat, choose a date range and a plotting backend, then run all cells.

1. **NDVI**: raw NDVI on every S2 date (clouds included), the cloud-masked NDVI, and the Whittaker-smoothed curve (lambda and d are parameters).
2. **Sentinel-1**: VH and VV of every track, with the rain that fell in the 24 h before each acquisition at this pixel (bars), plus daily rain (gray).
3. **S2 chips**: B5/B3/B2 chips (stretched) on every date where this pixel is cloud-free.

Everything heavy is precomputed (`scripts/build_stacks.py`, `scripts/precompute.py`), so a pixel loads in about 1 s.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "../scripts")
import importlib, explorer as ex
importlib.reload(ex)

## Parameters

In [ ]:
PIXEL_ID = 1348570          # pixel_id from data/grid/index.tif (row * 2219 + col); set to None to use LON/LAT
LON, LAT = None, None       # e.g. 103.31, 14.45 (used only when PIXEL_ID is None)
RADIUS = 0                  # 0 = single pixel; 2 = 5x5 window mean (S1 averaged in linear power)

START, END = "2026-04-01", "2026-10-08"   # date range to plot (YYYY-MM-DD)
BACKEND = "plotly"          # "plotly" (interactive) or "matplotlib" (static)

LAMBDA, D = 1.0, 2          # Whittaker smoothing: lambda and difference order, fitted on a daily grid
RAIN_SOURCE = "imerg"       # rain 24 h before each S1 date: "imerg" (NASA) or "gsmap" (JAXA, gauge-corrected)
SHOW_DAILY_RAIN = True      # gray bars: daily rain (CHIRPS to 2026-08-31, IMERG after)

CHIP_SIZE = 64              # chip size in pixels (64 px = 640 m)
CHIP_BANDS = ("B5", "B3", "B2")
STRETCH = "fixed"           # "fixed" = same AOI-wide 2-98 % stretch for all dates; "chip" = stretch each chip
CHIP_COLUMNS = 6

## Load the pixel

In [ ]:
px = ex.load_pixel(pixel_id=PIXEL_ID, lon=LON, lat=LAT, radius=RADIUS)
print(ex._title(px))
s2 = px["s2"]
print(f"S2 dates: {len(s2)}, cloud-free at this pixel: {int(s2.clear.sum())}   |   "
      f"S1 acquisitions: " + ", ".join(f"{t} {n}" for t, n in px["s1"].track.value_counts().items()))

## 1. NDVI: raw, cloud-masked, Whittaker

In [ ]:
ex.plot_ndvi(px, START, END, backend=BACKEND, lam=LAMBDA, d=D);

## 2. Sentinel-1 VH / VV with rain

In [ ]:
ex.plot_s1(px, START, END, backend=BACKEND, rain=RAIN_SOURCE, show_daily_rain=SHOW_DAILY_RAIN);

## 3. Sentinel-2 chips on cloud-free dates (red box = the pixel)

In [ ]:
ex.plot_chips(px, START, END, size=CHIP_SIZE, bands=CHIP_BANDS, stretch=STRETCH, ncols=CHIP_COLUMNS, backend=BACKEND);

## Tables (optional)

In [ ]:
from IPython.display import display
display(ex._clip(px["s2"], START, END).round(3).set_index("date").T)
display(ex._clip(px["s1"], START, END).round(2))

## Data notes

| Item | Detail |
|---|---|
| Grid | EPSG:32648, 10 m, 2219 x 2224 px; `pixel_id = row * 2219 + col`; band 2 of `index.tif` = inside AOI |
| Sentinel-2 | L2A harmonized, orbit 18 (tiles 48PTA/PTB/PUA/PUB mosaicked per date), 97 dates, B2 B3 B4 B5 B8 |
| Cloud-free rule | Cloud Score+ cs_cdf >= 0.60 and SCL not in {0, 1, 3, 8, 9, 10} and MSK_CLASSI_OPAQUE = 0 and MSK_CLASSI_CIRRUS = 0 |
| Whittaker | weighted Whittaker (Eilers 2003) on a daily grid, weight 1 on cloud-free dates, 0 elsewhere |
| Sentinel-1 | GRD IW, VV/VH in dB, terrain-corrected by Earth Engine, no speckle filter; RO164 descending (70 dates, ~33 deg), RO099 ascending (49 dates, ~43 deg) |
| Rain | IMERG V07 / GSMaP v8 at 0.1 deg (24 h before each S1 time), CHIRPS daily 0.05 deg (to 2026-08-31), IMERG daily after |
